### Healthcare Provider AI/ML Use Case on Databricks

Hospitals lose significant revenue due to 30-day patient readmissions while patient outcomes decline because high-risk patients are not identified before discharge.

Healthcare providers need an intelligent platform that can:

Predict patients likely to be readmitted
Identify the factors causing readmission
Recommend preventive interventions
Reduce hospital costs
Improve patient satisfaction

Traditional reporting only shows historical data; it does not predict future risk.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random

spark = SparkSession.builder.getOrCreate()

In [0]:
%pip install faker

import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()

In [0]:
num_records = 100000

In [0]:
genders = ["Male", "Female"]

diseases = [
    "Diabetes",
    "Hypertension",
    "Heart Disease",
    "Asthma",
    "Kidney Disease",
    "Cancer",
    "None"
]

departments = [
    "Cardiology",
    "Orthopedics",
    "Neurology",
    "General Medicine",
    "Oncology",
    "Emergency"
]

insurance = [
    "Private",
    "Government",
    "Self Pay"
]

In [0]:
import random

data = []

for i in range(num_records):

    age = random.randint(18,90)

    bmi = __builtins__.round(random.uniform(18,40),1)

    previous_admission = random.randint(0,8)

    medication_count = random.randint(1,15)

    lab_score = __builtins__.round(random.uniform(40,100),2)

    length_of_stay = random.randint(1,15)

    chronic = random.choice(diseases)

    heart_rate = random.randint(55,130)

    systolic_bp = random.randint(90,180)

    hba1c = __builtins__.round(random.uniform(4.5,11.5),1)

    followup = random.choice([0,1])

    emergency_visits = random.randint(0,6)

    missed_appointments = random.randint(0,5)

    patient = {

        "Patient_ID": i+1,

        "Age": age,

        "Gender": random.choice(genders),

        "BMI": bmi,

        "Chronic_Disease": chronic,

        "Department": random.choice(departments),

        "Insurance": random.choice(insurance),

        "Previous_Admissions": previous_admission,

        "Medication_Count": medication_count,

        "Lab_Result_Score": lab_score,

        "Length_of_Stay": length_of_stay,

        "Heart_Rate": heart_rate,

        "Systolic_BP": systolic_bp,

        "HbA1c": hba1c,

        "Emergency_Visits": emergency_visits,

        "Missed_Appointments": missed_appointments,

        "Follow_Up": followup

    }

    risk = (
        previous_admission*10
        + medication_count*2
        + length_of_stay*3
        + emergency_visits*5
        + missed_appointments*6
        + (15 if chronic!="None" else 0)
        + (10 if hba1c>7 else 0)
    )

    patient["Readmitted_30Days"] = 1 if risk>60 else 0

    data.append(patient)

In [0]:
import pandas as pd

pdf = pd.DataFrame(data)

pdf.head()

In [0]:
df = spark.createDataFrame(pdf)

In [0]:
df.write \
  .mode("overwrite") \
  .saveAsTable("provider_readmission")

In [0]:
from pyspark.sql.functions import *

In [0]:
# Source path (uploaded CSV files)
source_path = "/FileStore/healthcare/provider_readmission.csv"

# Schema location for Auto Loader
schema_path = "/FileStore/checkpoints/provider_schema"

# Checkpoint location
checkpoint_path = "/FileStore/checkpoints/provider_bronze"

# Bronze Delta location
bronze_path = "/FileStore/delta/provider_bronze"

In [0]:
bronze_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("cloudFiles.schemaLocation", schema_path)
         .option("header", "true")
         .load(source_path)
)

In [0]:
# Read from the table instead of streaming from DBFS
df_table = spark.table("provider_readmission")
df_table.printSchema()

In [0]:
# Since DBFS root is disabled, the data is already available in the provider_readmission table.
# If streaming is needed, use Unity Catalog Volumes for checkpoint and output locations.
# For now, reading from the existing table:
display(spark.table("provider_readmission"))

In [0]:
%sql
SELECT
COUNT(*) AS Total_Records,

SUM(CASE WHEN Patient_ID IS NULL THEN 1 ELSE 0 END) AS Missing_ID,

SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS Missing_Age,

SUM(CASE WHEN BMI IS NULL THEN 1 ELSE 0 END) AS Missing_BMI
FROM provider_readmission;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
bronze_df = spark.table("provider_bronze")

In [0]:
silver_df = bronze_df.dropDuplicates(["Patient_ID"])

In [0]:
from pyspark.sql.functions import col, sum

# Read from the correct table and apply deduplication
silver_df = spark.table("provider_readmission").dropDuplicates(["Patient_ID"])

silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
]).show()

In [0]:
silver_df = silver_df.fillna({
    "BMI": 25,
    "Lab_Result_Score": 70,
    "Heart_Rate": 75,
    "Systolic_BP": 120,
    "HbA1c": 5.5,
    "Medication_Count": 0,
    "Previous_Admissions": 0,
    "Emergency_Visits": 0,
    "Missed_Appointments": 0
})

In [0]:
silver_df = silver_df.fillna({
    "Gender": "Unknown",
    "Department": "General Medicine",
    "Insurance": "Unknown",
    "Chronic_Disease": "None"
})

In [0]:
silver_df = (
    silver_df
    .withColumn("Gender", initcap(trim(col("Gender"))))
    .withColumn("Department", initcap(trim(col("Department"))))
    .withColumn("Insurance", initcap(trim(col("Insurance"))))
    .withColumn("Chronic_Disease", initcap(trim(col("Chronic_Disease"))))
)

In [0]:
silver_df = (
    silver_df
    .withColumn("Patient_ID", col("Patient_ID").cast(IntegerType()))
    .withColumn("Age", col("Age").cast(IntegerType()))
    .withColumn("BMI", col("BMI").cast(DoubleType()))
    .withColumn("Previous_Admissions", col("Previous_Admissions").cast(IntegerType()))
    .withColumn("Medication_Count", col("Medication_Count").cast(IntegerType()))
    .withColumn("Lab_Result_Score", col("Lab_Result_Score").cast(DoubleType()))
    .withColumn("Length_of_Stay", col("Length_of_Stay").cast(IntegerType()))
    .withColumn("Heart_Rate", col("Heart_Rate").cast(IntegerType()))
    .withColumn("Systolic_BP", col("Systolic_BP").cast(IntegerType()))
    .withColumn("HbA1c", col("HbA1c").cast(DoubleType()))
    .withColumn("Emergency_Visits", col("Emergency_Visits").cast(IntegerType()))
    .withColumn("Missed_Appointments", col("Missed_Appointments").cast(IntegerType()))
    .withColumn("Follow_Up", col("Follow_Up").cast(IntegerType()))
    .withColumn("Readmitted_30Days", col("Readmitted_30Days").cast(IntegerType()))
)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("provider_silver")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
silver_df = spark.table("provider_silver")

display(silver_df)

In [0]:
gold_df = silver_df.withColumn(
    "Age_Group",
    when(col("Age") < 30, "Young")
    .when(col("Age") < 50, "Adult")
    .when(col("Age") < 70, "Middle_Aged")
    .otherwise("Senior")
)

In [0]:
gold_df = gold_df.withColumn(
    "BMI_Category",
    when(col("BMI") < 18.5, "Underweight")
    .when(col("BMI") < 25, "Normal")
    .when(col("BMI") < 30, "Overweight")
    .otherwise("Obese")
)

In [0]:
gold_df = gold_df.withColumn(
    "Chronic_Disease_Flag",
    when(col("Chronic_Disease") != "None", 1)
    .otherwise(0)
)

In [0]:
gold_df = gold_df.withColumn(
    "High_BP_Flag",
    when(col("Systolic_BP") >= 140, 1)
    .otherwise(0)
)

In [0]:
# Create missing flag columns first
gold_df = gold_df.withColumn(
    "Previous_Admission_Flag",
    when(col("Previous_Admissions") > 0, 1).otherwise(0)
).withColumn(
    "High_HbA1c_Flag",
    when(col("HbA1c") > 6.5, 1).otherwise(0)
).withColumn(
    "Emergency_Visit_Flag",
    when(col("Emergency_Visits") > 0, 1).otherwise(0)
).withColumn(
    "Missed_Appointment_Flag",
    when(col("Missed_Appointments") > 0, 1).otherwise(0)
).withColumn(
    "High_Medication_Flag",
    when(col("Medication_Count") > 5, 1).otherwise(0)
).withColumn(
    "Long_Stay_Flag",
    when(col("Length_of_Stay") > 7, 1).otherwise(0)
)

# Now calculate Risk_Score using the flags
gold_df = gold_df.withColumn(
    "Risk_Score",
    (
        col("Previous_Admission_Flag") * 2
        + col("Chronic_Disease_Flag") * 2
        + col("High_BP_Flag")
        + col("High_HbA1c_Flag") * 2
        + col("Emergency_Visit_Flag") * 2
        + col("Missed_Appointment_Flag")
        + col("High_Medication_Flag")
        + col("Long_Stay_Flag") * 2
    )
)

In [0]:
gold_df = gold_df.withColumn(
    "Risk_Category",
    when(col("Risk_Score") >= 8, "High")
    .when(col("Risk_Score") >= 4, "Medium")
    .otherwise("Low")
)

In [0]:
gold_df = gold_df.withColumn(
    "Lab_Risk_Flag",
    when(col("Lab_Result_Score") < 60, 1)
    .otherwise(0)
)

In [0]:
ml_df = gold_df.select(
    "Patient_ID",
    "Age",
    "BMI",
    "Previous_Admissions",
    "Medication_Count",
    "Lab_Result_Score",
    "Length_of_Stay",
    "Heart_Rate",
    "Systolic_BP",
    "HbA1c",
    "Emergency_Visits",
    "Missed_Appointments",
    "Follow_Up",

    "Chronic_Disease_Flag",
    "High_BP_Flag",
    "High_HbA1c_Flag",
    "Previous_Admission_Flag",
    "Emergency_Visit_Flag",
    "Missed_Appointment_Flag",
    "High_Medication_Flag",
    "Long_Stay_Flag",
    "Lab_Risk_Flag",

    "Risk_Score",
    "Risk_Category",

    "Readmitted_30Days"
)